Upload do dataset, primeiros passos e análise de qualidade dos dados.

In [1]:
from pathlib import Path

import pandas as pd


DATA_PATH = Path("../data/raw/transactional-sample.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

Dataset carregado: 3,199 linhas x 8 colunas


In [2]:
df.head()

,transaction_id,merchant_id,user_id,card_number,transaction_date,transaction_amount,device_id,has_cbk
0,21320398,29744,97051,434505******9116,2019-12-01T23:16:32.812632,374.56,285475.0,False
1,21320399,92895,2708,444456******4210,2019-12-01T22:45:37.873639,734.87,497105.0,True
2,21320400,47759,14777,425850******7024,2019-12-01T22:22:43.021495,760.36,NaN,False
3,21320401,68657,69758,464296******3991,2019-12-01T21:59:19.797129,2556.13,NaN,True
4,21320402,54075,64367,650487******6116,2019-12-01T21:30:53.347051,55.36,860232.0,False


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3199 entries, 0 to 3198
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   transaction_id      3199 non-null   int64  
 1   merchant_id         3199 non-null   int64  
 2   user_id             3199 non-null   int64  
 3   card_number         3199 non-null   str    
 4   transaction_date    3199 non-null   str    
 5   transaction_amount  3199 non-null   float64
 6   device_id           2369 non-null   float64
 7   has_cbk             3199 non-null   bool   
dtypes: bool(1), float64(2), int64(3), str(2)
memory usage: 178.2 KB


In [4]:
print("Colunas:")
for column in df.columns:
    print(f"- {column}")

Colunas:
- transaction_id
- merchant_id
- user_id
- card_number
- transaction_date
- transaction_amount
- device_id
- has_cbk


In [5]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
})

missing.sort_values("missing_count", ascending=False)

,missing_count,missing_pct
device_id,830,25.945608
transaction_id,0,0.000000
user_id,0,0.000000
merchant_id,0,0.000000
card_number,0,0.000000
transaction_date,0,0.000000
transaction_amount,0,0.000000
has_cbk,0,0.000000


In [6]:
print(f"Linhas duplicadas completas: {df.duplicated().sum():,}")

Linhas duplicadas completas: 0


In [7]:
id_columns = [
    "transaction_id",
    "merchant_id",
    "user_id",
    "card_number",
    "device_id",
]

for column in id_columns:
    print(
        f"{column}: "
        f"{df[column].nunique(dropna=True):,} únicos | "
        f"{df[column].duplicated().sum():,} duplicados"
    )

transaction_id: 3,199 únicos | 0 duplicados
merchant_id: 1,756 únicos | 1,443 duplicados
user_id: 2,704 únicos | 495 duplicados
card_number: 2,925 únicos | 274 duplicados
device_id: 1,996 únicos | 1,202 duplicados


In [8]:
print(df["has_cbk"].value_counts(dropna=False))
print()
print(df["has_cbk"].value_counts(normalize=True, dropna=False).mul(100).round(2))

has_cbk
False    2808
True      391
Name: count, dtype: int64

has_cbk
False    87.78
True     12.22
Name: proportion, dtype: float64


In [9]:
df["transaction_amount"].describe()

count    3199.000000
mean      767.812904
std       889.095904
min         1.220000
25%       205.235000
50%       415.940000
75%       981.680000
max      4097.210000
Name: transaction_amount, dtype: float64

In [10]:
print("Valores nulos:", df["transaction_amount"].isna().sum())
print("Valores <= 0:", (df["transaction_amount"] <= 0).sum())
print("Valores negativos:", (df["transaction_amount"] < 0).sum())

Valores nulos: 0
Valores <= 0: 0
Valores negativos: 0


In [11]:
transaction_dates = pd.to_datetime(
    df["transaction_date"],
    errors="coerce"
)

print("Datas inválidas:", transaction_dates.isna().sum())
print("Data mínima:", transaction_dates.min())
print("Data máxima:", transaction_dates.max())

Datas inválidas: 0
Data mínima: 2019-11-01 01:27:15.811098
Data máxima: 2019-12-01 23:16:32.812632


In [12]:
print(transaction_dates.describe())

count                          3199
mean     2019-11-22 12:47:02.242350
min      2019-11-01 01:27:15.811098
25%      2019-11-18 18:35:57.557813
50%      2019-11-23 13:50:58.758108
75%      2019-11-28 21:51:06.055557
max      2019-12-01 23:16:32.812632
Name: transaction_date, dtype: object


In [13]:
print("Devices únicos:", df["device_id"].nunique())
print("Transactions sem device:", df["device_id"].isna().sum())
print("Transactions com device:", df["device_id"].notna().sum())

Devices únicos: 1996
Transactions sem device: 830
Transactions com device: 2369


In [14]:
df["device_id"].value_counts(dropna=False).head(10)

device_id
NaN         830
563499.0     22
342890.0     19
101848.0     17
438940.0     14
547440.0     13
274282.0      8
223682.0      7
589318.0      7
542535.0      7
Name: count, dtype: int64